In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from joblib import dump
import os

print("=== 肿瘤抗原预测模型训练（使用59个精选特征） ===")

class TumorFeatureExtractor:
    def __init__(self, acid_file_path='acid.xlsx'):
        # 定义要提取的59个特征名称
        self.selected_features = [
            'sequence_length', 'P9_Hydrophobicity', 'P2_L', 'total_MW', 'P2_Hydrophobicity',
            'L_composition', 'total_pI', 'total_Hydrophobicity', 'avg_Hydrophobicity',
            'P9_MW', 'AA2', 'AA9', 'P2_MW', 'P2_pI', 'P9_plo', 'P2_plo', 'P9_pI',
            'P9_L', 'avg_MW', 'avg_plo', 'std_pI', 'P6_Hydrophobicity', 'avg_pI',
            'P1_pI', 'P4_pI', 'std_Hydrophobicity', 'std_MW', 'std_ele',
            'P4_Hydrophobicity', 'P1_MW', 'total_plo', 'P3_pI', 'P9_V', 'avg_ele',
            'P3_MW', 'AA1', 'P1_Hydrophobicity', 'P8_MW', 'P7_MW', 'AA6',
            'P3_Hydrophobicity', 'AA8', 'P8_pI', 'AA3', 'P7_pI', 'P6_MW',
            'P8_Hydrophobicity', 'P5_MW', 'AA7', 'P4_ele', 'AA5', 'AA4',
            'P6_pI', 'P5_Hydrophobicity', 'P5_pI', 'P4_MW', 'P7_Hydrophobicity',
            'std_Hydrocapto', 'P1_ele'
        ]
        self.feature_names = self.selected_features.copy()
        
        self.amino_acid_encoding = {
            'A': 1, 'R': 2, 'N': 3, 'D': 4, 'C': 5, 'Q': 6, 'E': 7, 'G': 8, 'H': 9, 'I': 10,
            'L': 11, 'K': 12, 'M': 13, 'F': 14, 'P': 15, 'S': 16, 'T': 17, 'W': 18, 'Y': 19, 'V': 20
        }
        
        # 加载氨基酸物化性质表
        self.acid_properties = self._load_acid_properties(acid_file_path)
        print(f"成功加载 {len(self.acid_properties)} 种氨基酸的物化性质")
    
    def _load_acid_properties(self, file_path):
        try:
            df = pd.read_excel(file_path)
            df.columns = [col.strip() for col in df.columns]
            df['acid'] = df['acid'].astype(str).str.strip()
            
            properties_dict = {}
            for _, row in df.iterrows():
                acid = row['acid']
                properties_dict[acid] = {
                    'hydrophobicity': row['Hydrophobicity'],
                    'pI': row['pI'],
                    'MW': row['MW'],
                    'ele': row['ele'],
                    'plo': row['plo'],
                    'hydrocapto': row['Hydrocapto']
                }
            return properties_dict
        except Exception as e:
            print(f"加载氨基酸性质文件失败: {e}")
            return {}
    
    def extract_features(self, sequences):
        if not sequences:
            raise ValueError("序列列表为空")
        
        print(f"正在为 {len(sequences)} 个序列提取59个精选特征...")
        features_list = []
        
        for i, seq in enumerate(sequences):
            if i % 1000 == 0 and i > 0:
                print(f"已处理 {i}/{len(sequences)} 个序列")
            
            features = self._extract_selected_features(seq)
            features_list.append(features)
        
        feature_df = pd.DataFrame(features_list, columns=self.selected_features)
        print(f"特征提取完成，共生成 {len(self.selected_features)} 个特征")
        return feature_df
    
    def _extract_selected_features(self, sequence):
        """只提取选定的59个特征"""
        features = {}
        sequence_upper = sequence.upper()
        
        # 1. 序列长度
        features['sequence_length'] = len(sequence)
        
        # 计算氨基酸组成
        total_length = len(sequence)
        L_count = sequence_upper.count('L')
        features['L_composition'] = L_count / total_length if total_length > 0 else 0
        
        # 计算氨基酸编码特征 (AA1-AA9)
        encoded = [self.amino_acid_encoding.get(aa.upper(), 0) for aa in sequence[:9]]
        encoded += [0] * (9 - len(encoded))
        for i in range(9):
            features[f'AA{i+1}'] = encoded[i]
        
        # 初始化物化性质值列表
        hydrophobicity_values = []
        pi_values = []
        mw_values = []
        ele_values = []
        plo_values = []
        hydrocapto_values = []
        
        # 收集每个氨基酸的物化性质
        for aa in sequence_upper:
            if aa in self.acid_properties:
                props = self.acid_properties[aa]
                hydrophobicity_values.append(props['hydrophobicity'])
                pi_values.append(props['pI'])
                mw_values.append(props['MW'])
                ele_values.append(props['ele'])
                plo_values.append(props['plo'])
                hydrocapto_values.append(props['hydrocapto'])
        
        # 计算统计特征
        if hydrophobicity_values:
            features['avg_Hydrophobicity'] = np.mean(hydrophobicity_values)
            features['std_Hydrophobicity'] = np.std(hydrophobicity_values)
            features['total_Hydrophobicity'] = np.sum(hydrophobicity_values)
        
        if pi_values:
            features['avg_pI'] = np.mean(pi_values)
            features['std_pI'] = np.std(pi_values)
            features['total_pI'] = np.sum(pi_values)
        
        if mw_values:
            features['avg_MW'] = np.mean(mw_values)
            features['std_MW'] = np.std(mw_values)
            features['total_MW'] = np.sum(mw_values)
        
        if ele_values:
            features['avg_ele'] = np.mean(ele_values)
            features['std_ele'] = np.std(ele_values)
        
        if plo_values:
            features['avg_plo'] = np.mean(plo_values)
            features['total_plo'] = np.sum(plo_values)
        
        if hydrocapto_values:
            features['std_Hydrocapto'] = np.std(hydrocapto_values)
        
        # 位置特异性特征
        # P2_L
        features['P2_L'] = 1 if len(sequence) > 1 and sequence[1].upper() == 'L' else 0
        
        # P9_L 和 P9_V
        features['P9_L'] = 1 if len(sequence) > 8 and sequence[8].upper() == 'L' else 0
        features['P9_V'] = 1 if len(sequence) > 8 and sequence[8].upper() == 'V' else 0
        
        # 位置特异性物化特征 (P1-P9)
        for i in range(9):
            if i < len(sequence_upper):
                aa = sequence_upper[i]
                if aa in self.acid_properties:
                    props = self.acid_properties[aa]
                    pos = i + 1
                    
                    # 根据特征列表选择性地提取特定位置的特征
                    if f'P{pos}_Hydrophobicity' in self.selected_features:
                        features[f'P{pos}_Hydrophobicity'] = props['hydrophobicity']
                    if f'P{pos}_pI' in self.selected_features:
                        features[f'P{pos}_pI'] = props['pI']
                    if f'P{pos}_MW' in self.selected_features:
                        features[f'P{pos}_MW'] = props['MW']
                    if f'P{pos}_ele' in self.selected_features:
                        features[f'P{pos}_ele'] = props['ele']
                    if f'P{pos}_plo' in self.selected_features:
                        features[f'P{pos}_plo'] = props['plo']
                else:
                    # 如果氨基酸不在表中，设置默认值
                    pos = i + 1
                    if f'P{pos}_Hydrophobicity' in self.selected_features:
                        features[f'P{pos}_Hydrophobicity'] = 0
                    if f'P{pos}_pI' in self.selected_features:
                        features[f'P{pos}_pI'] = 0
                    if f'P{pos}_MW' in self.selected_features:
                        features[f'P{pos}_MW'] = 0
                    if f'P{pos}_ele' in self.selected_features:
                        features[f'P{pos}_ele'] = 0
                    if f'P{pos}_plo' in self.selected_features:
                        features[f'P{pos}_plo'] = 0
            else:
                # 序列长度不足，设置默认值
                pos = i + 1
                if f'P{pos}_Hydrophobicity' in self.selected_features:
                    features[f'P{pos}_Hydrophobicity'] = 0
                if f'P{pos}_pI' in self.selected_features:
                    features[f'P{pos}_pI'] = 0
                if f'P{pos}_MW' in self.selected_features:
                    features[f'P{pos}_MW'] = 0
                if f'P{pos}_ele' in self.selected_features:
                    features[f'P{pos}_ele'] = 0
                if f'P{pos}_plo' in self.selected_features:
                    features[f'P{pos}_plo'] = 0
        
        # 确保所有选定的特征都有值
        for feature in self.selected_features:
            if feature not in features:
                features[feature] = 0
        
        return features
    
    def get_feature_names(self):
        return self.feature_names

def main():
    # 检查文件
    if not os.path.exists('acid.xlsx'):
        print("错误: 未找到 acid.xlsx 文件")
        return
    
    if not os.path.exists('A0201_160000.xlsx'):
        print("错误: 未找到 A0201_160000.xlsx 文件")
        return
    
    try:
        # 1. 加载数据
        print("1. 加载训练数据...")
        dataset = pd.read_excel('A0201_160000.xlsx', na_filter=False)
        print(f"成功加载 {len(dataset)} 条序列")
        print(f"正样本: {sum(dataset['label'])}, 负样本: {len(dataset) - sum(dataset['label'])}")
        
        # 2. 提取特征
        print("2. 提取59个精选特征...")
        extractor = TumorFeatureExtractor('acid.xlsx')
        feature_df = extractor.extract_features(dataset['sequence'].tolist())
        X = feature_df.values
        y = dataset['label'].values
        
        print(f"特征矩阵形状: {X.shape}")
        print(f"总特征数量: {len(extractor.feature_names)}")
        
        # 3. 数据划分
        print("\n3. 划分训练集和测试集...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
        
        # 4. 特征标准化
        print("4. 特征标准化...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # 5. 训练模型
        print("5. 训练模型...")
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        # 6. 评估模型
        print("6. 模型评估...")
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)
        
        print(f"测试准确率: {accuracy:.4f}")
        print(f"测试AUC: {auc:.4f}")
        
        print("\n分类报告:")
        print(classification_report(y_test, y_pred))
        
        print("\n混淆矩阵:")
        print(confusion_matrix(y_test, y_pred))
        
        # 7. 保存模型和预处理对象
        print("7. 保存模型...")
        dump(model, 'tumor_model_59_features.joblib')
        dump(scaler, 'tumor_scaler_59_features.joblib')
        
        # 保存特征名称
        with open('feature_names_59.txt', 'w') as f:
            for name in extractor.feature_names:
                f.write(name + '\n')
        
        print("模型训练完成!")
        print(f"模型文件: tumor_model_59_features.joblib")
        print(f"标准化器: tumor_scaler_59_features.joblib") 
        print(f"特征名称: feature_names_59.txt")
        print(f"使用的特征数量: {len(extractor.feature_names)}")
        
    except Exception as e:
        print(f"训练过程中出错: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

=== 肿瘤抗原预测模型训练（使用59个精选特征） ===
1. 加载训练数据...
成功加载 167642 条序列
正样本: 50308, 负样本: 117334
2. 提取59个精选特征...
成功加载 20 种氨基酸的物化性质
正在为 167642 个序列提取59个精选特征...
已处理 1000/167642 个序列
已处理 2000/167642 个序列
已处理 3000/167642 个序列
已处理 4000/167642 个序列
已处理 5000/167642 个序列
已处理 6000/167642 个序列
已处理 7000/167642 个序列
已处理 8000/167642 个序列
已处理 9000/167642 个序列
已处理 10000/167642 个序列
已处理 11000/167642 个序列
已处理 12000/167642 个序列
已处理 13000/167642 个序列
已处理 14000/167642 个序列
已处理 15000/167642 个序列
已处理 16000/167642 个序列
已处理 17000/167642 个序列
已处理 18000/167642 个序列
已处理 19000/167642 个序列
已处理 20000/167642 个序列
已处理 21000/167642 个序列
已处理 22000/167642 个序列
已处理 23000/167642 个序列
已处理 24000/167642 个序列
已处理 25000/167642 个序列
已处理 26000/167642 个序列
已处理 27000/167642 个序列
已处理 28000/167642 个序列
已处理 29000/167642 个序列
已处理 30000/167642 个序列
已处理 31000/167642 个序列
已处理 32000/167642 个序列
已处理 33000/167642 个序列
已处理 34000/167642 个序列
已处理 35000/167642 个序列
已处理 36000/167642 个序列
已处理 37000/167642 个序列
已处理 38000/167642 个序列
已处理 39000/167642 个序列
已处理 40000/167642 个序列
已处理 41000/167642 个序列
已处理 

In [2]:
import pandas as pd
import numpy as np
from joblib import load
import os

print("=== 肿瘤抗原预测（使用59个精选特征） ===")

class TumorFeatureExtractor:
    def __init__(self, acid_file_path='acid.xlsx'):
        # 定义59个精选特征名称（与训练时完全一致）
        self.selected_features = [
            'sequence_length', 'P9_Hydrophobicity', 'P2_L', 'total_MW', 'P2_Hydrophobicity',
            'L_composition', 'total_pI', 'total_Hydrophobicity', 'avg_Hydrophobicity',
            'P9_MW', 'AA2', 'AA9', 'P2_MW', 'P2_pI', 'P9_plo', 'P2_plo', 'P9_pI',
            'P9_L', 'avg_MW', 'avg_plo', 'std_pI', 'P6_Hydrophobicity', 'avg_pI',
            'P1_pI', 'P4_pI', 'std_Hydrophobicity', 'std_MW', 'std_ele',
            'P4_Hydrophobicity', 'P1_MW', 'total_plo', 'P3_pI', 'P9_V', 'avg_ele',
            'P3_MW', 'AA1', 'P1_Hydrophobicity', 'P8_MW', 'P7_MW', 'AA6',
            'P3_Hydrophobicity', 'AA8', 'P8_pI', 'AA3', 'P7_pI', 'P6_MW',
            'P8_Hydrophobicity', 'P5_MW', 'AA7', 'P4_ele', 'AA5', 'AA4',
            'P6_pI', 'P5_Hydrophobicity', 'P5_pI', 'P4_MW', 'P7_Hydrophobicity',
            'std_Hydrocapto', 'P1_ele'
        ]
        self.feature_names = self.selected_features.copy()
        
        self.amino_acid_encoding = {
            'A': 1, 'R': 2, 'N': 3, 'D': 4, 'C': 5, 'Q': 6, 'E': 7, 'G': 8, 'H': 9, 'I': 10,
            'L': 11, 'K': 12, 'M': 13, 'F': 14, 'P': 15, 'S': 16, 'T': 17, 'W': 18, 'Y': 19, 'V': 20
        }
        
        # 加载氨基酸物化性质表
        self.acid_properties = self._load_acid_properties(acid_file_path)
        print(f"成功加载 {len(self.acid_properties)} 种氨基酸的物化性质")
    
    def _load_acid_properties(self, file_path):
        try:
            df = pd.read_excel(file_path)
            df.columns = [col.strip() for col in df.columns]
            df['acid'] = df['acid'].astype(str).str.strip()
            
            properties_dict = {}
            for _, row in df.iterrows():
                acid = row['acid']
                properties_dict[acid] = {
                    'hydrophobicity': row['Hydrophobicity'],
                    'pI': row['pI'],
                    'MW': row['MW'],
                    'ele': row['ele'],
                    'plo': row['plo'],
                    'hydrocapto': row['Hydrocapto']
                }
            return properties_dict
        except Exception as e:
            print(f"加载氨基酸性质文件失败: {e}")
            return {}
    
    def extract_features(self, sequences):
        if not sequences:
            raise ValueError("序列列表为空")
        
        print(f"正在为 {len(sequences)} 个序列提取59个精选特征...")
        features_list = []
        
        for i, seq in enumerate(sequences):
            features = self._extract_selected_features(seq)
            features_list.append(features)
        
        feature_df = pd.DataFrame(features_list, columns=self.selected_features)
        print(f"特征提取完成，共生成 {len(self.selected_features)} 个特征")
        return feature_df
    
    def _extract_selected_features(self, sequence):
        """提取选定的59个特征"""
        features = {}
        sequence_upper = sequence.upper()
        
        # 1. 序列长度
        features['sequence_length'] = len(sequence)
        
        # 计算氨基酸组成 (L_composition)
        total_length = len(sequence)
        L_count = sequence_upper.count('L')
        features['L_composition'] = L_count / total_length if total_length > 0 else 0
        
        # 计算氨基酸编码特征 (AA1-AA9)
        encoded = [self.amino_acid_encoding.get(aa.upper(), 0) for aa in sequence[:9]]
        encoded += [0] * (9 - len(encoded))
        
        # 只提取需要的AA特征 (AA1, AA2, AA3, AA4, AA5, AA6, AA7, AA8, AA9)
        for i in range(9):
            features[f'AA{i+1}'] = encoded[i]
        
        # 初始化物化性质值列表
        hydrophobicity_values = []
        pi_values = []
        mw_values = []
        ele_values = []
        plo_values = []
        hydrocapto_values = []
        
        # 收集每个氨基酸的物化性质
        for aa in sequence_upper:
            if aa in self.acid_properties:
                props = self.acid_properties[aa]
                hydrophobicity_values.append(props['hydrophobicity'])
                pi_values.append(props['pI'])
                mw_values.append(props['MW'])
                ele_values.append(props['ele'])
                plo_values.append(props['plo'])
                hydrocapto_values.append(props['hydrocapto'])
        
        # 计算统计特征
        if hydrophobicity_values:
            features['avg_Hydrophobicity'] = np.mean(hydrophobicity_values)
            features['std_Hydrophobicity'] = np.std(hydrophobicity_values)
            features['total_Hydrophobicity'] = np.sum(hydrophobicity_values)
        else:
            features['avg_Hydrophobicity'] = 0
            features['std_Hydrophobicity'] = 0
            features['total_Hydrophobicity'] = 0
        
        if pi_values:
            features['avg_pI'] = np.mean(pi_values)
            features['std_pI'] = np.std(pi_values)
            features['total_pI'] = np.sum(pi_values)
        else:
            features['avg_pI'] = 0
            features['std_pI'] = 0
            features['total_pI'] = 0
        
        if mw_values:
            features['avg_MW'] = np.mean(mw_values)
            features['std_MW'] = np.std(mw_values)
            features['total_MW'] = np.sum(mw_values)
        else:
            features['avg_MW'] = 0
            features['std_MW'] = 0
            features['total_MW'] = 0
        
        if ele_values:
            features['avg_ele'] = np.mean(ele_values)
            features['std_ele'] = np.std(ele_values)
        else:
            features['avg_ele'] = 0
            features['std_ele'] = 0
        
        if plo_values:
            features['avg_plo'] = np.mean(plo_values)
            features['total_plo'] = np.sum(plo_values)
        else:
            features['avg_plo'] = 0
            features['total_plo'] = 0
        
        if hydrocapto_values:
            features['std_Hydrocapto'] = np.std(hydrocapto_values)
        else:
            features['std_Hydrocapto'] = 0
        
        # 位置特异性特征
        # P2_L
        features['P2_L'] = 1 if len(sequence) > 1 and sequence[1].upper() == 'L' else 0
        
        # P9_L 和 P9_V
        features['P9_L'] = 1 if len(sequence) > 8 and sequence[8].upper() == 'L' else 0
        features['P9_V'] = 1 if len(sequence) > 8 and sequence[8].upper() == 'V' else 0
        
        # 位置特异性物化特征
        for i in range(9):
            pos = i + 1
            if i < len(sequence_upper):
                aa = sequence_upper[i]
                if aa in self.acid_properties:
                    props = self.acid_properties[aa]
                    
                    # 根据特征列表选择性地提取特定位置的特征
                    if f'P{pos}_Hydrophobicity' in self.selected_features:
                        features[f'P{pos}_Hydrophobicity'] = props['hydrophobicity']
                    if f'P{pos}_pI' in self.selected_features:
                        features[f'P{pos}_pI'] = props['pI']
                    if f'P{pos}_MW' in self.selected_features:
                        features[f'P{pos}_MW'] = props['MW']
                    if f'P{pos}_ele' in self.selected_features:
                        features[f'P{pos}_ele'] = props['ele']
                    if f'P{pos}_plo' in self.selected_features:
                        features[f'P{pos}_plo'] = props['plo']
                else:
                    # 如果氨基酸不在表中，用0填充
                    if f'P{pos}_Hydrophobicity' in self.selected_features:
                        features[f'P{pos}_Hydrophobicity'] = 0
                    if f'P{pos}_pI' in self.selected_features:
                        features[f'P{pos}_pI'] = 0
                    if f'P{pos}_MW' in self.selected_features:
                        features[f'P{pos}_MW'] = 0
                    if f'P{pos}_ele' in self.selected_features:
                        features[f'P{pos}_ele'] = 0
                    if f'P{pos}_plo' in self.selected_features:
                        features[f'P{pos}_plo'] = 0
            else:
                # 如果序列长度不足，用0填充
                if f'P{pos}_Hydrophobicity' in self.selected_features:
                    features[f'P{pos}_Hydrophobicity'] = 0
                if f'P{pos}_pI' in self.selected_features:
                    features[f'P{pos}_pI'] = 0
                if f'P{pos}_MW' in self.selected_features:
                    features[f'P{pos}_MW'] = 0
                if f'P{pos}_ele' in self.selected_features:
                    features[f'P{pos}_ele'] = 0
                if f'P{pos}_plo' in self.selected_features:
                    features[f'P{pos}_plo'] = 0
        
        # 确保所有选定的特征都有值
        for feature in self.selected_features:
            if feature not in features:
                features[feature] = 0
        
        # 按照选定特征的顺序返回值
        return [features[feature] for feature in self.selected_features]

def main():
    # 检查必要文件
    required_files = [
        'tumor_model_59_features.joblib',
        'tumor_scaler_59_features.joblib',
        'acid.xlsx'
    ]
    
    missing = [f for f in required_files if not os.path.exists(f)]
    if missing:
        print(f"错误: 缺失文件 {missing}")
        return
    
    try:
        # 1. 加载模型和预处理对象
        print("1. 加载模型...")
        scaler = load('tumor_scaler_59_features.joblib')
        model = load('tumor_model_59_features.joblib')
        
        # 2. 加载预测数据
        input_file = 'A0201_100000_test.xlsx'
        output_file = "prediction_results_59_features.csv"
        
        print("2. 加载预测数据...")
        new_data = pd.read_excel(input_file)
        print(f"成功加载 {len(new_data)} 条序列")
        
        # 3. 提取59个特征
        print("3. 提取59个精选特征...")
        extractor = TumorFeatureExtractor('acid.xlsx')
        feature_df = extractor.extract_features(new_data['sequence'].tolist())
        X_new = feature_df.values
        
        print(f"特征矩阵形状: {X_new.shape}")
        
        # 4. 标准化和预测
        print("4. 进行预测...")
        X_new_scaled = scaler.transform(X_new)
        predictions_prob = model.predict_proba(X_new_scaled)[:, 1]
        predictions_label = (predictions_prob > 0.5).astype(int)
        
        # 5. 保存结果
        print("5. 保存结果...")
        result_df = pd.DataFrame({
            'sequence': new_data['sequence'],
            'prediction_score': predictions_prob,
            'prediction': predictions_label
        })
        result_df.to_csv(output_file, index=False)
        
        # 6. 输出统计
        positive_count = result_df['prediction'].sum()
        print(f"\n预测完成!")
        print(f"总序列数: {len(result_df)}")
        print(f"阳性预测数: {positive_count}")
        print(f"阳性比例: {positive_count/len(result_df)*100:.2f}%")
        print(f"结果已保存到: {output_file}")
        
    except Exception as e:
        print(f"预测过程中出错: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

=== 肿瘤抗原预测（使用59个精选特征） ===
1. 加载模型...
2. 加载预测数据...
成功加载 100000 条序列
3. 提取59个精选特征...
成功加载 20 种氨基酸的物化性质
正在为 100000 个序列提取59个精选特征...
特征提取完成，共生成 59 个特征
特征矩阵形状: (100000, 59)
4. 进行预测...
5. 保存结果...

预测完成!
总序列数: 100000
阳性预测数: 20306
阳性比例: 20.31%
结果已保存到: prediction_results_59_features.csv
